# PPO Reinforcement Learning for Cat-Qubit Stabilization

This notebook uses a compact **PPO-style** reinforcement-learning loop to optimize the cat-qubit controls `g_2` and `epsilon_d` **directly** from the physics model in the challenge notebooks.

The intended workflow is:

1. start from a physics-informed seed taken from the tutorial notebook conventions
2. represent the policy as a small Gaussian distribution over the **absolute physical controls** `g_2` and `epsilon_d`
3. sample a batch of candidate parameter vectors each epoch
4. evaluate every candidate with the **same cat-qubit Hamiltonian and loss channels** used in the challenge resources notebook
5. compute a scalar reward from physical metrics such as `T_X`, `T_Z`, fidelity, leakage, instability, bias, and distance from the physics seed
6. update the Gaussian mean and standard deviations with a clipped PPO objective plus entropy regularization

The code stays deliberately lightweight because this is a **low-dimensional continuous-control** problem, not a large neural-network RL benchmark.

## PPO scheme used here

The RL-control paper emphasizes a policy that continuously steers physical controls using observed error signals and a stable PPO-style update. For this low-dimensional setting, the analogous design is:

- **policy**: a Gaussian over the 4 real control coordinates `Re(g_2)`, `Im(g_2)`, `Re(epsilon_d)`, `Im(epsilon_d)`
- **state surrogate**: a physics-informed search seed based on the challenge notebook defaults
- **action**: one sampled control vector, interpreted directly as the physical parameters
- **environment**: a cat-qubit simulator built from the same Hamiltonian and dissipation model used in the challenge notebooks
- **reward**: a scalar score built from `T_X`, `T_Z`, fidelity, leakage, instability, bias, and distance from the physics seed
- **update**: clipped PPO objective with normalized advantages, entropy regularization, and a small replay buffer for recent samples

Notebook roadmap:

1. set up a GPU-aware `jax` / `dynamiqs` environment
2. rebuild the cat-qubit simulator in the notation of the resources notebook
3. measure `T_X` and `T_Z` from logical decay curves
4. define a PPO-compatible reward
5. train a compact Gaussian policy to search over the physical controls directly
6. visualize decay curves, reward improvement, policy evolution, and physics diagnostics

In [ ]:
from __future__ import annotations

import os
from collections import deque
from dataclasses import dataclass
from typing import Callable, Protocol

# This helps keep GPU memory use friendlier when running many simulations in a notebook.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import dynamiqs as dq
import jax
import jax.numpy as jnp
import numpy as np
from jax import jit, value_and_grad
from matplotlib import pyplot as plt
from scipy.optimize import least_squares

# If needed, install dependencies from the repo root with:
# !pip install -r ../requirements.txt

gpu_devices = [device for device in jax.devices() if device.platform == "gpu"]
print("JAX devices:", jax.devices())
if gpu_devices:
    print("GPU simulation is available. dynamiqs/JAX will use the GPU-backed runtime.")
else:
    print("No GPU detected. The notebook will still run, but on CPU.")

In [ ]:
@dataclass(frozen=True)
class SearchSeedParams:
    epsilon_d: complex
    g2: complex

    def to_vector(self) -> jnp.ndarray:
        return jnp.array(
            [
                float(np.real(self.g2)),
                float(np.imag(self.g2)),
                float(np.real(self.epsilon_d)),
                float(np.imag(self.epsilon_d)),
            ],
            dtype=jnp.float32,
        )

    @classmethod
    def from_vector(cls, vector: jnp.ndarray) -> "SearchSeedParams":
        vector = jnp.asarray(vector, dtype=jnp.float32)
        return cls(
            epsilon_d=complex(float(vector[2]), float(vector[3])),
            g2=complex(float(vector[0]), float(vector[1])),
        )


@dataclass(frozen=True)
class ParameterBounds:
    lower: jnp.ndarray
    upper: jnp.ndarray

    @classmethod
    def default_from_existing_optimizer(cls) -> "ParameterBounds":
        # Same safe search region used by the existing optimizer in this repo.
        return cls(
            lower=jnp.array([0.25, -1.5, 0.5, -4.0], dtype=jnp.float32),
            upper=jnp.array([2.0, 1.5, 8.0, 4.0], dtype=jnp.float32),
        )

    @property
    def span(self) -> jnp.ndarray:
        return self.upper - self.lower

    def clip(self, vector: jnp.ndarray) -> jnp.ndarray:
        return jnp.clip(jnp.asarray(vector, dtype=jnp.float32), self.lower, self.upper)


@dataclass(frozen=True)
class SimulationConfig:
    na: int = 15
    nb: int = 5
    kappa_a: float = 1.0
    kappa_b: float = 10.0
    x_tfinal: float = 2.0
    z_tfinal: float = 80.0
    nsave: int = 64


@dataclass(frozen=True)
class RewardConfig:
    target_bias: float = 25.0
    w_tx: float = 1.0
    w_tz: float = 1.0
    w_bias_penalty: float = 0.30
    w_fidelity: float = 1.50
    w_leakage: float = 1.50
    w_instability: float = 0.50
    w_distance: float = 0.35
    w_physics_penalty: float = 0.75


@dataclass(frozen=True)
class PPOConfig:
    batch_size: int = 6
    replay_capacity: int = 48
    replay_sample_size: int = 12
    ppo_epochs: int = 6
    clip_ratio: float = 0.20
    entropy_coef: float = 0.02
    learning_rate: float = 0.03
    initial_std: float = 0.18
    min_std: float = 0.02
    max_std: float = 0.55
    seed: int = 7


@dataclass(frozen=True)
class CandidateBatch:
    actions: jnp.ndarray
    parameters: jnp.ndarray
    log_probs: jnp.ndarray


@dataclass(frozen=True)
class EvaluationBatch:
    actions: jnp.ndarray
    parameters: jnp.ndarray
    log_probs: jnp.ndarray
    rewards: jnp.ndarray
    metrics: dict[str, jnp.ndarray]


def unpack_controls(vector: jnp.ndarray) -> tuple[complex, complex]:
    vector = jnp.asarray(vector, dtype=jnp.float32)
    g2 = complex(float(vector[0]), float(vector[1]))
    epsilon_d = complex(float(vector[2]), float(vector[3]))
    return g2, epsilon_d


def scaled_distance_from_seed(
    parameter_batch: jnp.ndarray,
    seed_vector: jnp.ndarray,
    bounds: ParameterBounds,
) -> jnp.ndarray:
    scaled = (parameter_batch - seed_vector) / jnp.maximum(bounds.span, 1e-6)
    return jnp.linalg.norm(scaled, axis=-1)


def physics_informed_seed() -> SearchSeedParams:
    # These are the tutorial-style values used in the challenge notebook examples.
    return SearchSeedParams(epsilon_d=4.0 + 0.0j, g2=1.0 + 0.0j)


bounds = ParameterBounds.default_from_existing_optimizer()
sim_config = SimulationConfig()
seed_params = physics_informed_seed()
print("Bounds span:", bounds.span)
print("Simulation config:", sim_config)
print("Physics-informed seed:", seed_params)

## Physics model from the resources notebook

This notebook stays aligned with `challenge/resources/Tutorial__Introduction_to_Cats.ipynb` and the cat-stabilization section of `challenge/1-challenge.ipynb`.

The simulator below uses the same ingredients:

- storage mode `a` and lossy buffer mode `b`
- buffer-coupled Hamiltonian
  
  `H / hbar = g_2^* a^2 b^dagger + g_2 (a^dagger)^2 b - epsilon_d b^dagger - epsilon_d^* b`
- loss channels
  
  `L_b = sqrt(kappa_b) b`, `L_a = sqrt(kappa_a) a`
- effective cat-qubit relations
  
  `kappa_2 = 4 |g_2|^2 / kappa_b`, `eps_2 = 2 g_2 epsilon_d / kappa_b`

To keep the PPO search physically interpretable, every candidate is annotated with:

- estimated cat size `|alpha|`
- effective two-photon rate `kappa_2`
- fast-buffer ratio `kappa_b / max(|g_2|, |epsilon_d|)`
- a simple physics penalty for leaving a comfortable cat-size or fast-buffer regime

In [ ]:
def monoexp_model(params: np.ndarray, time: np.ndarray) -> np.ndarray:
    amplitude, tau, offset = params
    return amplitude * np.exp(-time / tau) + offset


def robust_exp_fit(time: np.ndarray, values: np.ndarray) -> dict[str, np.ndarray | float]:
    time = np.asarray(time, dtype=float)
    values = np.asarray(values, dtype=float)

    amplitude0 = max(abs(values[0] - values[-1]), 1e-4)
    tau0 = max(float(time[-1] - time[0]) / 3.0, 1e-3)
    offset0 = float(values[-1])

    def residuals(params: np.ndarray) -> np.ndarray:
        return monoexp_model(params, time) - values

    result = least_squares(
        residuals,
        x0=np.array([amplitude0, tau0, offset0], dtype=float),
        bounds=([0.0, 1e-6, -1.5], [5.0, np.inf, 1.5]),
        loss="soft_l1",
        f_scale=0.05,
    )
    fit_curve = monoexp_model(result.x, time)
    rmse = float(np.sqrt(np.mean((fit_curve - values) ** 2)))

    return {
        "params": result.x,
        "tau": float(result.x[1]),
        "fit_curve": fit_curve,
        "rmse": rmse,
    }


def reference_cat_amplitude(
    g2: complex,
    epsilon_d: complex,
    *,
    kappa_a: float,
    kappa_b: float,
) -> tuple[complex, complex, float]:
    kappa_2 = max(4.0 * abs(g2) ** 2 / kappa_b, 1e-6)
    eps_2 = 2.0 * g2 * epsilon_d / kappa_b
    alpha_sq = (2.0 * eps_2 / kappa_2) - (kappa_a / (2.0 * kappa_2))
    alpha = complex(jnp.sqrt(jnp.asarray(alpha_sq, dtype=jnp.complex64)))

    if abs(alpha) < 0.5:
        alpha = 0.5 + 0.0j

    return alpha, eps_2, kappa_2


def normalized_cat_state(na: int, alpha: complex, parity: str) -> dq.QArray:
    plus = dq.coherent(na, alpha)
    minus = dq.coherent(na, -alpha)
    if parity == "even":
        return dq.unit(plus + minus)
    if parity == "odd":
        return dq.unit(plus - minus)
    raise ValueError(f"Unknown cat parity: {parity}")


def build_logical_operators(na: int, alpha: complex) -> dict[str, dq.QArray]:
    a_storage = dq.destroy(na)
    plus_z = dq.coherent(na, alpha)
    minus_z = dq.coherent(na, -alpha)
    plus_x = normalized_cat_state(na, alpha, "even")
    minus_x = normalized_cat_state(na, alpha, "odd")

    x_operator = (1j * jnp.pi * a_storage.dag() @ a_storage).expm()
    z_operator = plus_z @ plus_z.dag() - minus_z @ minus_z.dag()
    code_projector = plus_x @ plus_x.dag() + minus_x @ minus_x.dag()

    return {
        "plus_z": plus_z,
        "minus_z": minus_z,
        "plus_x": plus_x,
        "minus_x": minus_x,
        "x_operator": x_operator,
        "z_operator": z_operator,
        "code_projector": code_projector,
    }


def fast_buffer_diagnostics(
    g2: complex,
    epsilon_d: complex,
    *,
    alpha_abs: float,
    kappa_b: float,
) -> dict[str, float]:
    fast_buffer_ratio = float(kappa_b / max(abs(g2), abs(epsilon_d), 1e-6))
    alpha_penalty = max(0.0, 0.75 - alpha_abs) ** 2 + max(0.0, alpha_abs - 3.25) ** 2
    fast_buffer_penalty = max(0.0, 5.0 - fast_buffer_ratio) ** 2
    return {
        "fast_buffer_ratio": fast_buffer_ratio,
        "physics_penalty": float(alpha_penalty + fast_buffer_penalty),
    }


print("Physics seed uses the tutorial-style starting point g2=1 and epsilon_d=4.")

## Candidate evaluation with the real cat-qubit model

The next cell evaluates one candidate control vector by running the storage-plus-buffer dynamics twice:

- once from a logical `+z` state to estimate the `T_Z` decay curve
- once from a logical `+x` state to estimate the `T_X` decay curve

From those two trajectories we extract:

- `T_X`
- `T_Z`
- bias `T_Z / T_X`
- a codespace-fidelity proxy from the cat-subspace population
- leakage `1 - fidelity`
- instability from the fit residuals
- distance from a physics-informed seed derived from the tutorial values
- cat-physics diagnostics such as `|alpha|`, `kappa_2`, and the fast-buffer ratio

For PPO training we then wrap the single-candidate evaluator in a simple batched interface.

In [ ]:
class BatchedSimulator(Protocol):
    def __call__(self, parameter_batch: jnp.ndarray) -> dict[str, jnp.ndarray]:
        ...


def simulate_logical_decay(
    initial_state: str,
    *,
    g2: complex,
    epsilon_d: complex,
    config: SimulationConfig,
) -> dict[str, object]:
    a_storage = dq.destroy(config.na)
    a = dq.tensor(a_storage, dq.eye(config.nb))
    b = dq.tensor(dq.eye(config.na), dq.destroy(config.nb))

    alpha_estimate, eps_2, kappa_2 = reference_cat_amplitude(
        g2,
        epsilon_d,
        kappa_a=config.kappa_a,
        kappa_b=config.kappa_b,
    )
    operators = build_logical_operators(config.na, alpha_estimate)

    hamiltonian = (
        jnp.conj(g2) * a @ a @ b.dag()
        + g2 * a.dag() @ a.dag() @ b
        - epsilon_d * b.dag()
        - jnp.conj(epsilon_d) * b
    )
    losses = [
        jnp.sqrt(config.kappa_b) * b,
        jnp.sqrt(config.kappa_a) * a,
    ]

    tfinal = config.z_tfinal if initial_state == "+z" else config.x_tfinal
    tsave = jnp.linspace(0.0, tfinal, config.nsave)

    state_map = {
        "+z": operators["plus_z"],
        "-z": operators["minus_z"],
        "+x": operators["plus_x"],
        "-x": operators["minus_x"],
    }
    psi0 = dq.tensor(state_map[initial_state], dq.fock(config.nb, 0))

    exp_ops = [
        dq.tensor(operators["x_operator"], dq.eye(config.nb)),
        dq.tensor(operators["z_operator"], dq.eye(config.nb)),
        dq.tensor(operators["code_projector"], dq.eye(config.nb)),
    ]

    result = dq.mesolve(
        hamiltonian,
        losses,
        psi0,
        tsave,
        options=dq.Options(progress_meter=False),
        exp_ops=exp_ops,
    )

    return {
        "time": np.asarray(result.tsave, dtype=float),
        "x_signal": np.asarray(result.expects[0].real, dtype=float),
        "z_signal": np.asarray(result.expects[1].real, dtype=float),
        "codespace_population": np.asarray(result.expects[2].real, dtype=float),
        "alpha_abs": float(abs(alpha_estimate)),
        "eps_2_abs": float(abs(eps_2)),
        "kappa_2": float(kappa_2),
    }


def evaluate_single_candidate(
    parameter_vector: jnp.ndarray,
    *,
    seed_vector: jnp.ndarray,
    bounds: ParameterBounds,
    config: SimulationConfig,
    return_curves: bool = False,
) -> dict[str, object]:
    parameter_vector = bounds.clip(parameter_vector)
    g2, epsilon_d = unpack_controls(parameter_vector)

    try:
        z_run = simulate_logical_decay(
            "+z",
            g2=g2,
            epsilon_d=epsilon_d,
            config=config,
        )
        x_run = simulate_logical_decay(
            "+x",
            g2=g2,
            epsilon_d=epsilon_d,
            config=config,
        )

        fit_z = robust_exp_fit(z_run["time"], z_run["z_signal"])
        fit_x = robust_exp_fit(x_run["time"], x_run["x_signal"])

        Tx = max(float(fit_x["tau"]), 1e-6)
        Tz = max(float(fit_z["tau"]), 1e-6)
        bias = Tz / Tx
        fidelity = float(0.5 * (x_run["codespace_population"][-1] + z_run["codespace_population"][-1]))
        leakage = max(0.0, 1.0 - fidelity)
        instability = 0.5 * (float(fit_x["rmse"]) + float(fit_z["rmse"]))
        distance = float(
            scaled_distance_from_seed(
                jnp.asarray(parameter_vector)[None, :],
                jnp.asarray(seed_vector),
                bounds,
            )[0]
        )

        physics = fast_buffer_diagnostics(
            g2,
            epsilon_d,
            alpha_abs=float(z_run["alpha_abs"]),
            kappa_b=config.kappa_b,
        )

        summary: dict[str, object] = {
            "Tx": Tx,
            "Tz": Tz,
            "bias": bias,
            "fidelity": fidelity,
            "leakage": leakage,
            "instability": instability,
            "distance_from_seed": distance,
            "alpha_abs": float(z_run["alpha_abs"]),
            "kappa_2": float(z_run["kappa_2"]),
            "eps_2_abs": float(z_run["eps_2_abs"]),
            "fast_buffer_ratio": physics["fast_buffer_ratio"],
            "physics_penalty": physics["physics_penalty"],
        }

        if return_curves:
            summary.update(
                {
                    "x_time": x_run["time"],
                    "x_signal": x_run["x_signal"],
                    "x_fit": fit_x["fit_curve"],
                    "x_codespace_population": x_run["codespace_population"],
                    "z_time": z_run["time"],
                    "z_signal": z_run["z_signal"],
                    "z_fit": fit_z["fit_curve"],
                    "z_codespace_population": z_run["codespace_population"],
                }
            )

        return summary
    except Exception as exc:  # pragma: no cover - defensive fallback for unstable candidates
        penalty = {
            "Tx": 1e-6,
            "Tz": 1e-6,
            "bias": 1.0,
            "fidelity": 0.0,
            "leakage": 1.0,
            "instability": 10.0,
            "distance_from_seed": float(
                scaled_distance_from_seed(
                    jnp.asarray(parameter_vector)[None, :],
                    jnp.asarray(seed_vector),
                    bounds,
                )[0]
            ),
            "alpha_abs": 0.0,
            "kappa_2": 0.0,
            "eps_2_abs": 0.0,
            "fast_buffer_ratio": 0.0,
            "physics_penalty": 25.0,
            "error": str(exc),
        }
        return penalty


def batched_candidate_evaluator(
    parameter_batch: jnp.ndarray,
    *,
    seed_vector: jnp.ndarray,
    bounds: ParameterBounds,
    config: SimulationConfig,
) -> dict[str, jnp.ndarray]:
    metric_keys = [
        "Tx",
        "Tz",
        "bias",
        "fidelity",
        "leakage",
        "instability",
        "distance_from_seed",
        "alpha_abs",
        "kappa_2",
        "eps_2_abs",
        "fast_buffer_ratio",
        "physics_penalty",
    ]
    collected: dict[str, list[float]] = {key: [] for key in metric_keys}

    for vector in np.asarray(parameter_batch, dtype=float):
        summary = evaluate_single_candidate(
            jnp.asarray(vector, dtype=jnp.float32),
            seed_vector=seed_vector,
            bounds=bounds,
            config=config,
            return_curves=False,
        )
        for key in metric_keys:
            collected[key].append(float(summary[key]))

    return {key: jnp.asarray(values, dtype=jnp.float32) for key, values in collected.items()}


preview = evaluate_single_candidate(
    seed_params.to_vector(),
    seed_vector=seed_params.to_vector(),
    bounds=bounds,
    config=sim_config,
    return_curves=False,
)
print({key: preview[key] for key in ["Tx", "Tz", "bias", "fidelity", "leakage", "alpha_abs", "kappa_2"]})

## PPO components

Because only four real control coordinates are being optimized, the policy does not need a neural network.

Instead we keep a compact Gaussian policy with:

- a trainable mean vector for the physical controls themselves
- a trainable log-standard-deviation vector

That is enough to implement the PPO ingredients we want:

- Gaussian action sampling directly in control space
- old and new log-probabilities
- clipped likelihood-ratio objective
- normalized advantages
- entropy regularization
- sample reuse through a short replay buffer

This keeps the implementation close in spirit to the paper while remaining readable and practical for low-dimensional continuous optimization.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int, seed: int = 0) -> None:
        self.capacity = capacity
        self.rng = np.random.default_rng(seed)
        self.storage: deque[dict[str, np.ndarray]] = deque(maxlen=capacity)

    def add_batch(self, evaluation: EvaluationBatch) -> None:
        metrics_np = {key: np.asarray(value) for key, value in evaluation.metrics.items()}
        batch_size = len(np.asarray(evaluation.rewards))
        for idx in range(batch_size):
            self.storage.append(
                {
                    "actions": np.asarray(evaluation.actions[idx]),
                    "parameters": np.asarray(evaluation.parameters[idx]),
                    "log_probs": np.asarray(evaluation.log_probs[idx]),
                    "rewards": np.asarray(evaluation.rewards[idx]),
                    **{key: value[idx] for key, value in metrics_np.items()},
                }
            )

    def sample(self, sample_size: int) -> EvaluationBatch | None:
        if not self.storage:
            return None

        size = min(sample_size, len(self.storage))
        indices = self.rng.choice(len(self.storage), size=size, replace=False)
        samples = [self.storage[int(index)] for index in indices]
        metric_keys = [key for key in samples[0] if key not in {"actions", "parameters", "log_probs", "rewards"}]

        return EvaluationBatch(
            actions=jnp.asarray(np.stack([sample["actions"] for sample in samples]), dtype=jnp.float32),
            parameters=jnp.asarray(np.stack([sample["parameters"] for sample in samples]), dtype=jnp.float32),
            log_probs=jnp.asarray(np.stack([sample["log_probs"] for sample in samples]), dtype=jnp.float32),
            rewards=jnp.asarray(np.stack([sample["rewards"] for sample in samples]), dtype=jnp.float32),
            metrics={
                key: jnp.asarray(np.stack([sample[key] for sample in samples]), dtype=jnp.float32)
                for key in metric_keys
            },
        )


@jit
def gaussian_log_prob(actions: jnp.ndarray, mean: jnp.ndarray, std: jnp.ndarray) -> jnp.ndarray:
    variance = jnp.square(std)
    centered = actions - mean
    return -0.5 * jnp.sum(jnp.square(centered) / variance + jnp.log(2.0 * jnp.pi * variance), axis=-1)


@jit
def gaussian_entropy(log_std: jnp.ndarray) -> jnp.ndarray:
    return jnp.sum(log_std + 0.5 * jnp.log(2.0 * jnp.pi * jnp.e))


@jit
def normalize_advantages(rewards: jnp.ndarray) -> jnp.ndarray:
    rewards = jnp.asarray(rewards, dtype=jnp.float32)
    return (rewards - rewards.mean()) / (rewards.std() + 1e-6)


def make_reward_function(config: RewardConfig) -> Callable[[dict[str, jnp.ndarray]], jnp.ndarray]:
    @jit
    def reward_fn(metrics: dict[str, jnp.ndarray]) -> jnp.ndarray:
        bias_penalty = jnp.square(jnp.log((metrics["bias"] + 1e-6) / config.target_bias))
        return (
            config.w_tx * jnp.log1p(metrics["Tx"])
            + config.w_tz * jnp.log1p(metrics["Tz"])
            - config.w_bias_penalty * bias_penalty
            + config.w_fidelity * metrics["fidelity"]
            - config.w_leakage * metrics["leakage"]
            - config.w_instability * metrics["instability"]
            - config.w_distance * metrics["distance_from_seed"]
            - config.w_physics_penalty * metrics["physics_penalty"]
        )

    return reward_fn


def ppo_loss(
    mean: jnp.ndarray,
    log_std: jnp.ndarray,
    actions: jnp.ndarray,
    old_log_probs: jnp.ndarray,
    advantages: jnp.ndarray,
    clip_ratio: float,
    entropy_coef: float,
) -> jnp.ndarray:
    std = jnp.exp(log_std)
    new_log_probs = gaussian_log_prob(actions, mean, std)
    ratios = jnp.exp(new_log_probs - old_log_probs)
    unclipped = ratios * advantages
    clipped = jnp.clip(ratios, 1.0 - clip_ratio, 1.0 + clip_ratio) * advantages
    surrogate = jnp.minimum(unclipped, clipped)
    return -(jnp.mean(surrogate) + entropy_coef * gaussian_entropy(log_std))


def concatenate_evaluations(primary: EvaluationBatch, secondary: EvaluationBatch | None) -> EvaluationBatch:
    if secondary is None:
        return primary

    metric_keys = primary.metrics.keys()
    return EvaluationBatch(
        actions=jnp.concatenate([primary.actions, secondary.actions], axis=0),
        parameters=jnp.concatenate([primary.parameters, secondary.parameters], axis=0),
        log_probs=jnp.concatenate([primary.log_probs, secondary.log_probs], axis=0),
        rewards=jnp.concatenate([primary.rewards, secondary.rewards], axis=0),
        metrics={
            key: jnp.concatenate([primary.metrics[key], secondary.metrics[key]], axis=0)
            for key in metric_keys
        },
    )


reward_config = RewardConfig()
reward_fn = make_reward_function(reward_config)
print("Reward combines Tx, Tz, bias, fidelity, leakage, instability, distance from the physics seed, and physics penalties.")

## The PPO refiner class

The main class below is the reusable RL module.

Its API is intentionally small:

- `sample_candidates()`
- `evaluate_candidates()`
- `train_step()`
- `get_best_parameters()`

Internally it:

1. starts from a physics-informed seed from the tutorial notebook
2. samples a batch of Gaussian actions directly in the physical control space
3. clips candidates to safe parameter bounds
4. evaluates them with the cat-qubit simulator
5. computes rewards from the chosen reward function
6. stores recent batches in a replay buffer
7. performs a few PPO epochs on the compact Gaussian policy

In [ ]:
class RLParameterRefiner:
    def __init__(
        self,
        *,
        seed_params: SearchSeedParams,
        simulator_fn: BatchedSimulator,
        reward_fn: Callable[[dict[str, jnp.ndarray]], jnp.ndarray],
        parameter_bounds: ParameterBounds,
        config: PPOConfig,
    ) -> None:
        self.seed_params = seed_params
        self.seed_vector = seed_params.to_vector()
        self.simulator_fn = simulator_fn
        self.reward_fn = reward_fn
        self.parameter_bounds = parameter_bounds
        self.config = config

        self.mean = self.seed_vector.astype(jnp.float32)
        self.log_std = jnp.log(jnp.full_like(self.seed_vector, config.initial_std))
        self.rng = jax.random.PRNGKey(config.seed)
        self.replay_buffer = ReplayBuffer(config.replay_capacity, seed=config.seed)
        self.history: list[dict[str, object]] = []
        self.best_record: dict[str, object] | None = None

    def _clip_policy_state(self) -> None:
        self.mean = jnp.clip(self.mean, self.parameter_bounds.lower, self.parameter_bounds.upper)
        self.log_std = jnp.clip(
            self.log_std,
            jnp.log(jnp.full_like(self.log_std, self.config.min_std)),
            jnp.log(jnp.full_like(self.log_std, self.config.max_std)),
        )

    def sample_candidates(self, batch_size: int | None = None) -> CandidateBatch:
        batch_size = batch_size or self.config.batch_size
        self.rng, subkey = jax.random.split(self.rng)
        noise = jax.random.normal(subkey, shape=(batch_size, self.mean.shape[0]))
        std = jnp.exp(self.log_std)
        actions = self.mean + std * noise
        parameters = self.parameter_bounds.clip(actions)
        log_probs = gaussian_log_prob(parameters, self.mean, std)
        return CandidateBatch(actions=parameters, parameters=parameters, log_probs=log_probs)

    def evaluate_candidates(self, candidates: CandidateBatch, *, store_in_replay: bool = True) -> EvaluationBatch:
        metrics = self.simulator_fn(candidates.parameters)
        rewards = self.reward_fn(metrics)
        evaluation = EvaluationBatch(
            actions=candidates.actions,
            parameters=candidates.parameters,
            log_probs=candidates.log_probs,
            rewards=jnp.asarray(rewards, dtype=jnp.float32),
            metrics={key: jnp.asarray(value, dtype=jnp.float32) for key, value in metrics.items()},
        )

        if store_in_replay:
            self.replay_buffer.add_batch(evaluation)

        self._update_best(evaluation)
        return evaluation

    def _update_best(self, evaluation: EvaluationBatch) -> None:
        rewards_np = np.asarray(evaluation.rewards, dtype=float)
        best_index = int(np.argmax(rewards_np))
        best_reward = float(rewards_np[best_index])

        if self.best_record is not None and best_reward <= float(self.best_record["reward"]):
            return

        best_vector = np.asarray(evaluation.parameters[best_index], dtype=float)
        best_params = SearchSeedParams.from_vector(jnp.asarray(best_vector, dtype=jnp.float32))
        self.best_record = {
            "reward": best_reward,
            "parameters_vector": best_vector,
            "g2": best_params.g2,
            "epsilon_d": best_params.epsilon_d,
            "metrics": {key: float(np.asarray(value)[best_index]) for key, value in evaluation.metrics.items()},
        }

    def _policy_update(self, evaluation: EvaluationBatch) -> dict[str, float]:
        advantages = normalize_advantages(evaluation.rewards)
        mean = self.mean
        log_std = self.log_std
        loss_value = None

        for _ in range(self.config.ppo_epochs):
            loss_value, grads = value_and_grad(ppo_loss, argnums=(0, 1))(
                mean,
                log_std,
                evaluation.actions,
                evaluation.log_probs,
                advantages,
                self.config.clip_ratio,
                self.config.entropy_coef,
            )
            mean = mean - self.config.learning_rate * grads[0]
            log_std = log_std - self.config.learning_rate * grads[1]
            self.mean = mean
            self.log_std = log_std
            self._clip_policy_state()
            mean = self.mean
            log_std = self.log_std

        std = jnp.exp(self.log_std)
        new_log_probs = gaussian_log_prob(evaluation.actions, self.mean, std)
        approx_kl = jnp.mean(evaluation.log_probs - new_log_probs)
        entropy = gaussian_entropy(self.log_std)

        return {
            "loss": float(loss_value),
            "approx_kl": float(approx_kl),
            "entropy": float(entropy),
        }

    def train_step(self, batch_size: int | None = None) -> dict[str, float]:
        candidates = self.sample_candidates(batch_size=batch_size)
        fresh_evaluation = self.evaluate_candidates(candidates, store_in_replay=True)
        replay_evaluation = self.replay_buffer.sample(self.config.replay_sample_size)
        update_batch = concatenate_evaluations(fresh_evaluation, replay_evaluation)
        update_stats = self._policy_update(update_batch)

        batch_rewards = np.asarray(fresh_evaluation.rewards, dtype=float)
        batch_best_index = int(np.argmax(batch_rewards))
        summary = {
            "reward_mean": float(jnp.mean(fresh_evaluation.rewards)),
            "reward_max": float(jnp.max(fresh_evaluation.rewards)),
            "Tx_mean": float(jnp.mean(fresh_evaluation.metrics["Tx"])),
            "Tz_mean": float(jnp.mean(fresh_evaluation.metrics["Tz"])),
            "fidelity_mean": float(jnp.mean(fresh_evaluation.metrics["fidelity"])),
            "leakage_mean": float(jnp.mean(fresh_evaluation.metrics["leakage"])),
            "bias_mean": float(jnp.mean(fresh_evaluation.metrics["bias"])),
            "best_reward": float(self.best_record["reward"]),
            "best_Tx": float(np.asarray(fresh_evaluation.metrics["Tx"])[batch_best_index]),
            "best_Tz": float(np.asarray(fresh_evaluation.metrics["Tz"])[batch_best_index]),
            "best_bias": float(np.asarray(fresh_evaluation.metrics["bias"])[batch_best_index]),
            **update_stats,
        }

        self.history.append(
            {
                **summary,
                "mean": np.asarray(self.mean, dtype=float),
                "std": np.asarray(jnp.exp(self.log_std), dtype=float),
                "sampled_parameters": np.asarray(fresh_evaluation.parameters, dtype=float),
                "sampled_rewards": np.asarray(fresh_evaluation.rewards, dtype=float),
                "sampled_Tx": np.asarray(fresh_evaluation.metrics["Tx"], dtype=float),
                "sampled_Tz": np.asarray(fresh_evaluation.metrics["Tz"], dtype=float),
                "sampled_bias": np.asarray(fresh_evaluation.metrics["bias"], dtype=float),
                "sampled_fidelity": np.asarray(fresh_evaluation.metrics["fidelity"], dtype=float),
                "sampled_leakage": np.asarray(fresh_evaluation.metrics["leakage"], dtype=float),
                "sampled_instability": np.asarray(fresh_evaluation.metrics["instability"], dtype=float),
                "sampled_alpha_abs": np.asarray(fresh_evaluation.metrics["alpha_abs"], dtype=float),
                "sampled_fast_buffer_ratio": np.asarray(fresh_evaluation.metrics["fast_buffer_ratio"], dtype=float),
            }
        )
        return summary

    def get_best_parameters(self) -> dict[str, object]:
        if self.best_record is None:
            raise RuntimeError("No candidate has been evaluated yet.")
        return self.best_record


preview_seed = physics_informed_seed()
preview_simulator = lambda parameter_batch: batched_candidate_evaluator(
    parameter_batch,
    seed_vector=preview_seed.to_vector(),
    bounds=bounds,
    config=sim_config,
)
preview_refiner = RLParameterRefiner(
    seed_params=preview_seed,
    simulator_fn=preview_simulator,
    reward_fn=reward_fn,
    parameter_bounds=bounds,
    config=PPOConfig(batch_size=2, replay_sample_size=2, ppo_epochs=2),
)
preview_candidates = preview_refiner.sample_candidates()
print("Preview candidate batch shape:", preview_candidates.parameters.shape)

## Training protocol

This notebook no longer depends on a teammate's CMA output.

Instead, PPO starts from a **physics-informed seed** taken from the tutorial notebooks and searches over the physical controls directly.

To keep the notebook practical, the default training run uses small batches and a modest number of epochs, since every candidate requires real `dynamiqs` simulations for both `T_X` and `T_Z` decay estimation.

In [ ]:
seed = physics_informed_seed()
seed_vector = seed.to_vector()

simulator_fn = lambda parameter_batch: batched_candidate_evaluator(
    parameter_batch,
    seed_vector=seed_vector,
    bounds=bounds,
    config=sim_config,
)

ppo_config = PPOConfig(
    batch_size=6,
    replay_capacity=48,
    replay_sample_size=12,
    ppo_epochs=6,
    clip_ratio=0.20,
    entropy_coef=0.02,
    learning_rate=0.03,
    initial_std=0.18,
    min_std=0.02,
    max_std=0.55,
    seed=13,
)

refiner = RLParameterRefiner(
    seed_params=seed,
    simulator_fn=simulator_fn,
    reward_fn=reward_fn,
    parameter_bounds=bounds,
    config=ppo_config,
)

num_epochs = 8
for epoch in range(num_epochs):
    metrics = refiner.train_step()
    print(
        f"Epoch {epoch:02d} | reward_mean={metrics['reward_mean']:.4f} | "
        f"reward_max={metrics['reward_max']:.4f} | best_reward={metrics['best_reward']:.4f} | "
        f"Tx_mean={metrics['Tx_mean']:.4f} | Tz_mean={metrics['Tz_mean']:.4f} | bias_mean={metrics['bias_mean']:.4f}"
    )

best = refiner.get_best_parameters()
print()
print("Best PPO-discovered parameters")
print("  seed      =", seed)
print("  g2        =", best["g2"])
print("  epsilon_d =", best["epsilon_d"])
print("  reward    =", best["reward"])
print("  metrics   =", best["metrics"])

## Characterization graphs

These figures are meant to answer both optimization and physics questions:

- Is PPO improving the reward over time?
- Are the policy mean and standard deviations moving in a stable way?
- Are the best `T_X` and `T_Z` values increasing?
- Is the sampled population moving toward a useful region of `g_2` and `epsilon_d`?
- Are fidelity, leakage, cat size, and fast-buffer diagnostics staying in a reasonable regime?

In [ ]:
def plot_training_summary(refiner: RLParameterRefiner) -> None:
    history = refiner.history
    epochs = np.arange(len(history))

    reward_mean = np.array([entry["reward_mean"] for entry in history], dtype=float)
    reward_max = np.array([entry["reward_max"] for entry in history], dtype=float)
    best_reward = np.array([entry["best_reward"] for entry in history], dtype=float)
    approx_kl = np.array([entry["approx_kl"] for entry in history], dtype=float)
    entropy = np.array([entry["entropy"] for entry in history], dtype=float)
    best_tx = np.array([entry["best_Tx"] for entry in history], dtype=float)
    best_tz = np.array([entry["best_Tz"] for entry in history], dtype=float)
    best_bias = np.array([entry["best_bias"] for entry in history], dtype=float)
    means = np.stack([entry["mean"] for entry in history])
    stds = np.stack([entry["std"] for entry in history])

    sampled_parameters = np.concatenate([entry["sampled_parameters"] for entry in history], axis=0)
    sampled_rewards = np.concatenate([entry["sampled_rewards"] for entry in history], axis=0)
    sampled_Tx = np.concatenate([entry["sampled_Tx"] for entry in history], axis=0)
    sampled_Tz = np.concatenate([entry["sampled_Tz"] for entry in history], axis=0)
    sampled_fidelity = np.concatenate([entry["sampled_fidelity"] for entry in history], axis=0)
    sampled_leakage = np.concatenate([entry["sampled_leakage"] for entry in history], axis=0)
    sampled_alpha_abs = np.concatenate([entry["sampled_alpha_abs"] for entry in history], axis=0)
    sampled_fast_buffer_ratio = np.concatenate([entry["sampled_fast_buffer_ratio"] for entry in history], axis=0)

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes[0, 0].plot(epochs, reward_mean, marker="o", label="Reward mean")
    axes[0, 0].plot(epochs, reward_max, marker="o", label="Reward max")
    axes[0, 0].plot(epochs, best_reward, marker="o", label="Best-so-far reward")
    axes[0, 0].set_title("PPO reward progression")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    labels = [r"Re($g_2$)", r"Im($g_2$)", r"Re($\epsilon_d$)", r"Im($\epsilon_d$)"]
    for idx, label in enumerate(labels):
        axes[0, 1].plot(epochs, means[:, idx], marker="o", label=f"mean {label}")
        axes[0, 1].plot(epochs, stds[:, idx], linestyle="--", alpha=0.8, label=f"std {label}")
    axes[0, 1].set_title("Policy mean and std")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend(ncol=2, fontsize=8)

    axes[1, 0].semilogy(epochs, best_tx, marker="o", label=r"$T_X$")
    axes[1, 0].semilogy(epochs, best_tz, marker="o", label=r"$T_Z$")
    axes[1, 0].set_title("Best decay times per epoch")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    axes[1, 1].plot(epochs, best_bias, marker="o", label=r"$T_Z / T_X$")
    axes[1, 1].plot(epochs, approx_kl, marker="o", label="Approx KL")
    axes[1, 1].plot(epochs, entropy, marker="o", label="Entropy")
    axes[1, 1].set_title("Bias and PPO diagnostics")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()
    fig.tight_layout()
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    sc0 = axes[0, 0].scatter(sampled_parameters[:, 0], sampled_parameters[:, 1], c=sampled_rewards, cmap="viridis", s=40)
    axes[0, 0].set_title(r"Sampled $g_2$ controls")
    axes[0, 0].set_xlabel(r"Re($g_2$)")
    axes[0, 0].set_ylabel(r"Im($g_2$)")
    axes[0, 0].grid(True, alpha=0.3)
    fig.colorbar(sc0, ax=axes[0, 0], label="Reward")

    sc1 = axes[0, 1].scatter(sampled_parameters[:, 2], sampled_parameters[:, 3], c=sampled_rewards, cmap="viridis", s=40)
    axes[0, 1].set_title(r"Sampled $\epsilon_d$ controls")
    axes[0, 1].set_xlabel(r"Re($\epsilon_d$)")
    axes[0, 1].set_ylabel(r"Im($\epsilon_d$)")
    axes[0, 1].grid(True, alpha=0.3)
    fig.colorbar(sc1, ax=axes[0, 1], label="Reward")

    sc2 = axes[1, 0].scatter(sampled_Tx, sampled_Tz, c=sampled_rewards, cmap="plasma", s=40)
    axes[1, 0].set_title(r"$T_X$ versus $T_Z$")
    axes[1, 0].set_xlabel(r"$T_X$")
    axes[1, 0].set_ylabel(r"$T_Z$")
    axes[1, 0].grid(True, alpha=0.3)
    fig.colorbar(sc2, ax=axes[1, 0], label="Reward")

    sc3 = axes[1, 1].scatter(sampled_fidelity, sampled_leakage, c=sampled_rewards, cmap="plasma", s=40)
    axes[1, 1].set_title("Fidelity versus leakage")
    axes[1, 1].set_xlabel("Fidelity")
    axes[1, 1].set_ylabel("Leakage")
    axes[1, 1].grid(True, alpha=0.3)
    fig.colorbar(sc3, ax=axes[1, 1], label="Reward")
    fig.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(sampled_alpha_abs, bins=20, alpha=0.85)
    axes[0].set_title("Estimated cat size |alpha|")
    axes[0].set_xlabel("|alpha|")
    axes[0].set_ylabel("Count")
    axes[0].grid(True, alpha=0.3)

    axes[1].hist(sampled_fast_buffer_ratio, bins=20, alpha=0.85)
    axes[1].set_title("Fast-buffer ratio")
    axes[1].set_xlabel(r"$\kappa_b / \max(|g_2|, |\epsilon_d|)$")
    axes[1].set_ylabel("Count")
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_training_summary(refiner)

## Decay curves for the best PPO solution

The most direct way to characterize whether the RL protocol is working is to inspect the actual logical decay curves of the best candidate.

The next cell re-simulates the best PPO-discovered controls with trace data enabled and plots:

- the logical `Z` decay with its exponential fit and extracted `T_Z`
- the logical `X` decay with its exponential fit and extracted `T_X`
- the cat-subspace population during both runs
- a small summary table of the key physical metrics

In [ ]:
best_details = evaluate_single_candidate(
    jnp.asarray(best["parameters_vector"], dtype=jnp.float32),
    seed_vector=seed_vector,
    bounds=bounds,
    config=sim_config,
    return_curves=True,
)


def plot_best_candidate_decay(details: dict[str, object]) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    axes[0, 0].plot(details["z_time"], details["z_signal"], label=r"Measured $\langle Z_L \rangle$")
    axes[0, 0].plot(details["z_time"], details["z_fit"], linestyle="--", label=r"Fit, $T_Z={:.2f}$".format(details["Tz"]))
    axes[0, 0].set_title(r"Logical $T_Z$ decay")
    axes[0, 0].set_xlabel("Time")
    axes[0, 0].set_ylabel(r"$\langle Z_L \rangle$")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    axes[0, 1].plot(details["x_time"], details["x_signal"], label=r"Measured $\langle X_L \rangle$")
    axes[0, 1].plot(details["x_time"], details["x_fit"], linestyle="--", label=r"Fit, $T_X={:.2f}$".format(details["Tx"]))
    axes[0, 1].set_title(r"Logical $T_X$ decay")
    axes[0, 1].set_xlabel("Time")
    axes[0, 1].set_ylabel(r"$\langle X_L \rangle$")
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    axes[1, 0].plot(details["z_time"], details["z_codespace_population"], label="+z run")
    axes[1, 0].plot(details["x_time"], details["x_codespace_population"], label="+x run")
    axes[1, 0].set_title("Cat-subspace population")
    axes[1, 0].set_xlabel("Time")
    axes[1, 0].set_ylabel("Codespace population")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    axes[1, 1].axis("off")
    summary_text = "\n".join(
        [
            f"seed = {seed}",
            f"g2 = {best['g2']}",
            f"epsilon_d = {best['epsilon_d']}",
            f"Reward = {best['reward']:.4f}",
            f"Tx = {details['Tx']:.4f}",
            f"Tz = {details['Tz']:.4f}",
            f"Bias = {details['bias']:.4f}",
            f"Fidelity = {details['fidelity']:.4f}",
            f"Leakage = {details['leakage']:.4f}",
            f"Distance from seed = {details['distance_from_seed']:.4f}",
            f"|alpha| = {details['alpha_abs']:.4f}",
            f"kappa_2 = {details['kappa_2']:.4f}",
            f"Fast-buffer ratio = {details['fast_buffer_ratio']:.4f}",
        ]
    )
    axes[1, 1].text(0.02, 0.98, summary_text, va="top", ha="left", family="monospace")

    fig.tight_layout()
    plt.show()


plot_best_candidate_decay(best_details)

print("This PPO run starts from the tutorial-inspired seed and searches directly over physical g2 and epsilon_d values.")